# Día 4B · Chunking estructural

Compara el chunking recursivo actual con una variante estructural usando exclusivamente BGE-M3 normal.

In [ ]:
!git clone -q https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git
%cd proyecto_ActividadGrado-Riesgos
!git checkout -q dia-04b-chunking-estructural
!pip -q install pandas pyarrow pymupdf python-docx sentence-transformers openai scikit-learn

## 1. Construir chunks estructurales
La variante conserva documento y página, prioriza títulos, párrafos y listas, y limita cada fragmento a 210 palabras.

In [ ]:
from pathlib import Path
import pandas as pd, shutil, zipfile
out=Path('/content/dia4b'); out.mkdir(exist_ok=True)
!python -m src.preprocessing.build_structural_chunks --inventory data/evaluation/inventario_corpus.csv --corpus-dir data/raw/corpus_actualizado --output-dir /content/dia4b --target-words 160 --max-words 210 --min-words 30
actual=pd.read_csv('data/processed/chunks/chunks_recursive.csv')
structural=pd.read_csv('/content/dia4b/chunks_structural.csv')
display(pd.DataFrame([{'estrategia':'actual','chunks':len(actual),'promedio_palabras':actual.chunk_size_words.mean(),'mediana':actual.chunk_size_words.median(),'maximo':actual.chunk_size_words.max()},{'estrategia':'estructural','chunks':len(structural),'promedio_palabras':structural.chunk_size_words.mean(),'mediana':structural.chunk_size_words.median(),'maximo':structural.chunk_size_words.max()}]))

## 2. Recuperar Top-5 con BGE-M3
Activa GPU T4 antes de ejecutar esta celda.

In [ ]:
!python -m src.retrieval.compare_structural_chunking --structural-chunks /content/dia4b/chunks_structural.csv --baseline-results data/processed/embedding/retrieval_results_bge_m3.csv --questions data/evaluation/gold_questions.csv --output-dir /content/dia4b --top-k 5

## 3. Subir el ZIP del Día 4A
Sube `resultados_reranking_multilingue_dia_04a.zip`. Se reutilizarán sus juicios y solo se evaluarán pares nuevos.

In [ ]:
from google.colab import files
uploaded=files.upload()
zip_name=next(n for n in uploaded if n.startswith('resultados_reranking_multilingue_dia_04a'))
checkpoint_dir=Path('/content/checkpoint4a'); checkpoint_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as z:z.extractall(checkpoint_dir)
checkpoint=checkpoint_dir/'judge_v3_checkpoint.jsonl'
assert checkpoint.exists(), 'El ZIP no contiene judge_v3_checkpoint.jsonl'
shutil.copy(checkpoint,out/'judge_v3_checkpoint.jsonl')
print('Checkpoint validado')

## 4. Evaluar relevancia
La clave se usa solo durante esta sesión y no se guarda.

In [ ]:
import os
from getpass import getpass
os.environ['OPENAI_API_KEY']=getpass('OPENAI_API_KEY: ')
!python -m src.retrieval.rejudge_hyde_results --results /content/dia4b/chunking_comparison_results.csv --questions data/evaluation/gold_questions.csv --output-dir /content/dia4b
metrics=pd.read_csv('/content/dia4b/metrics_by_method_v3.csv').sort_values('mrr',ascending=False)
display(metrics)

## 5. Descargar resultados

In [ ]:
result_zip=shutil.make_archive('/content/resultados_chunking_estructural_dia_04b','zip','/content/dia4b')
files.download(result_zip)